<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/RAGAS_Video_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Library + API keys

In [ ]:
!pip install --upgrade --quiet unstructured
!pip install --quiet langchain-community langchain-openai faiss-cpu
!pip install --quiet ragas

In [ ]:
!pip install --upgrade ragas instructor langchain-openai

In [ ]:
# urls for kubernetes documentation
urls = [
    "https://kubernetes.io/docs/concepts/overview/",
    "https://kubernetes.io/docs/concepts/architecture/",
    "https://kubernetes.io/docs/concepts/containers/",
    "https://kubernetes.io/docs/concepts/workloads/",
    "https://kubernetes.io/docs/concepts/storage/"
]

In [ ]:
# download NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
from google.colab import userdata
openai_api = userdata.get('OPENAI_API_KEY')

import os
os.environ['OPENAI_API_KEY'] = openai_api

MODEL = "gpt-5-nano"
EMBEDDING_MODEL = 'text-embedding-3-large'

# Data and RAG

In [ ]:
# Import libraries, functions, classes
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

In [ ]:
# Load the documents and split them
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 3000,
    chunk_overlap = 600
)

loader = UnstructuredURLLoader(urls = urls)
data = loader.load()

In [ ]:
documents = text_splitter.split_documents(documents = data)

In [ ]:
documents[0:1]

In [ ]:
# Create the embeddings and store in the Database vectorstor
embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(documents = documents, embedding = embeddings)

In [ ]:
# Perform RAG
def rag(query, k = 20):
  retrieved_docs = vectorstore.similarity_search(query, k = k)
  context = "\n".join([doc.page_content for doc in retrieved_docs])

  # Define the prompt
  prompt = f""" answer this query {query} based on the context {context}
  """

  # Call the LLM
  model = ChatOpenAI(
      model = MODEL
  )

  response = model.invoke(prompt)

  # Return the output
  return response.content, [txt.page_content for txt in retrieved_docs]

# Test the function
query = "Give me an overview of kubernetes"
answer, __ = rag(query)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(answer))

# Generate Synthethic Data

**6/28/2026:** Quick fix due to old `langchain_community.chat_models.vertexai` llm, link here: https://github.com/vibrantlabsai/ragas/issues/2753

In [ ]:
import sys
import types

dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [ ]:
# Import classes
from ragas.llms import llm_factory
from ragas.testset import TestsetGenerator
from openai import OpenAI
from ragas.embeddings.base import embedding_factory

In [ ]:
# Connect to OpenAI
client = OpenAI(api_key = openai_api)

In [ ]:
# LLMS and embeddings
generator_llm = llm_factory(
    model = "gpt-4o",
    client = client
)

In [ ]:
# Generator embeddings
generator_embeddings = embedding_factory(
    "openai",
    model = EMBEDDING_MODEL,
    client = client
)

In [ ]:
# Remove empty chunks
chunks = [doc for doc in documents if doc.page_content and doc.page_content.strip()]

# Test Set Generation
generator = TestsetGenerator(
    llm = generator_llm,
    embedding_model = generator_embeddings
)
dataset = generator.generate_with_chunks(
    chunks = chunks,
    testset_size = 30,
    raise_exceptions=False
)

In [ ]:
# Transform the dataset
test_df = dataset.to_pandas()
test_df.head()

In [ ]:
from tqdm import tqdm

# Answer each query and put it in our df
answer_gen = []
context_gen = []
for query in tqdm(test_df["user_input"], desc = "Generating answers from synthethic dataset...", total = len(test_df["user_input"])):
  answer, context = rag(query)
  answer_gen.append(answer)
  context_gen.append(context)

test_df["answer_gen"] = answer_gen
test_df["context_gen"] = context_gen
test_df.head()

In [ ]:
test_df.to_csv("synthethic_data.csv")

In [ ]:
!pwd

# Rouge Score

Ragas Documentation Rouge Score: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/traditional/?h=rouge#rouge-score

In [ ]:
!pip install --quiet rouge-score

In [ ]:
# Modern Version 6/29/2026
from ragas.metrics.collections import RougeScore

# Create metric (no LLM/embeddings needed)
scorer = RougeScore(rouge_type="rougeL", mode="fmeasure")

# Evaluate
result = await scorer.ascore(
    reference="The Eiffel Tower is located in Paris.",
    response="The Eiffel Tower is located in India."
)
print(f"ROUGE Score: {result.value}")

In [ ]:
# Simple sample of RougeScore
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import RougeScore

sample = SingleTurnSample(
    response = "The Eiffel Tower in located in India.",
    reference = "The Eiffel Tower in located in Paris."
)

scorer = RougeScore()
score = await scorer.single_turn_ascore(sample)
print(score)

In [ ]:
# Import classes and libraries
from ragas.metrics.collections import RougeScore
import numpy as np

# Create metric (no LLM/embeddings needed)
scorer = RougeScore(rouge_type="rougeL", mode="fmeasure")

In [ ]:
rouge_score_list = []
for row in test_df.to_dict(orient = "records"):
  rouge_score = await scorer.ascore(
      reference = row["reference"],
      response = row["answer_gen"]
  )
  rouge_score_list.append(rouge_score.value)

In [ ]:
print(f"First 5 rouge scores: {rouge_score_list[0:5]}")
print(f"The mean rouge score is: {np.mean(rouge_score_list)}")

In [ ]:
import ragas
print(ragas.__version__)

# LLM Based Evaluation

In [ ]:
# Define the evaluation llm and embedding model
from openai import AsyncOpenAI, OpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

# Connect to OpenAI
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

print(f"[INFO] Our embedding model is: {EMBEDDING_MODEL}")
eval_embeddings = embedding_factory(
    "openai",
    model = EMBEDDING_MODEL,
    client = client
)

## Simple Criteria Scoring
Simple Criteria Scoring RAGAS: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/general_purpose/#simple-criteria-scoring

In [ ]:
from ragas.metrics import DiscreteMetric

In [ ]:
# Define the simple scorer
simple_scorer = DiscreteMetric(
    name="simple scorer",
    allowed_values=list(range(0, 5)),  # 0 to 5
    prompt="""Score 0 to 5 by similarity between reference and generated answer

    The following are the keys:
    user_input : {user_input}
    reference : {reference}
    generated answer : {response}

    Respond with only the number (0-5).
    """,
)


In [ ]:
# Get the simple score for every row
simple_score_list = []
for row in test_df.to_dict(orient = "records"):
  simple_score = await simple_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      response = row["answer_gen"],
      llm = eval_llm
  )
  simple_score_list.append(simple_score.value)

In [ ]:
# Print the outputs
print(simple_score_list)
print(f"The mean simple score is: {np.mean(simple_score_list)}")

## Rubrics Based Scoring

Link: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/general_purpose/#rubrics-based-scoring

In [ ]:
rubrics = {
    "score1_description": "The response is entirely incorrect and fails to address any aspect of the reference.",
    "score2_description": "The response contains partial accuracy but includes major errors or significant omissions that affect its relevance to the reference.",
    "score3_description": "The response is mostly accurate but lacks clarity, thoroughness, or minor details needed to fully address the reference.",
    "score4_description": "The response is accurate and clear, with only minor omissions or slight inaccuracies in addressing the reference.",
    "score5_description": "The response is completely accurate, clear, and thoroughly addresses the reference without any errors or omissions.",
}

In [ ]:
from ragas.metrics.collections import RubricsScoreWithReference

In [ ]:
# Define the rubric Scorer
rubrics_scorer = RubricsScoreWithReference(
    rubrics=rubrics,
    prompt="""Using the rubrics passed, score the reference and generated answer

    The following are the keys:
    user_input : {user_input}
    reference : {reference}
    generated answer : {response}
    """,
    llm=eval_llm,
)


In [ ]:
# Compute the rubrics score for each row
rubrics_score_list = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Rubrics Score...", total = len(test_df.to_dict(orient = "records"))):
  rubrics_score = await rubrics_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      response = row["answer_gen"]
  )
  rubrics_score_list.append(rubrics_score.value)

In [ ]:
# Print the rubrics score using the score
print(rubrics_score_list)
print(f"The mean rubrics score is: {np.mean(rubrics_score_list)}")

# RAG-Specific Metrics

## Factual Correctness

In [ ]:
from ragas.metrics.collections import FactualCorrectness

In [ ]:
# Decfine the Factual Correcteness scorer
# Connect to OpenAI
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

# eval_llm = llm_factory(
#     model = "gpt-5-mini",
#     client = client,
#     max_tokens = 2048
# )

factual_scorer = FactualCorrectness(
    llm=eval_llm,
    atomicity="low",
    coverage="low")

In [ ]:
factual_scores_list = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Factual Score...", total = len(test_df.to_dict(orient = "records"))):
  factual_score = await factual_scorer.ascore(
      reference = row["reference"],
      response = row["answer_gen"]
  )
  factual_scores_list.append(factual_score.value)

In [ ]:
# Print the first 5 and mean
print(factual_scores_list[0:5])
print(f"The mean factual score is: {np.mean(factual_scores_list)}")

## Semantic Similarity

In [ ]:
from ragas.metrics.collections import SemanticSimilarity

In [ ]:
# Get the semantic scorer
semantic_scorer = SemanticSimilarity(
    embeddings = eval_embeddings)

In [ ]:
semantic_similarity_score = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Semantic Similarity Score...", total = len(test_df.to_dict(orient = "records"))):
  semantic_score = await semantic_scorer.ascore(
      reference = row["reference"],
      response = row["answer_gen"]
  )
  semantic_similarity_score.append(semantic_score.value)


In [ ]:
# Print the first 5 and mean
print(semantic_similarity_score[0:5])
print(f"The mean semantic similarity score is: {np.mean(semantic_similarity_score)}")

## Context Precision

In [ ]:
from ragas.metrics.collections import ContextPrecision

In [ ]:
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

In [ ]:
context_precision_scorer = ContextPrecision(llm = eval_llm)

In [ ]:
# Create a context precision list
context_precision_scores = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Context Precision Score...", total = len(test_df.to_dict(orient = "records"))):
  context_precision_score = await context_precision_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      retrieved_contexts=row["context_gen"])
  context_precision_scores.append(context_precision_score.value)

In [ ]:
# Print the first 5 and mean
print(context_precision_scores[0:5])
print(f"The mean context precision score is: {np.mean(context_precision_scores)}")

## Context Recall

In [ ]:
from ragas.metrics.collections import ContextRecall

In [ ]:
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client
)

In [ ]:
# Setup context recall scorer
context_recall_scorer = ContextRecall(llm = eval_llm)

In [ ]:
# Create a context recall list
context_recall_scores = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Context Recall Score...", total = len(test_df.to_dict(orient = "records"))):
  context_recall_score = await context_recall_scorer.ascore(
      user_input = row["user_input"],
      reference = row["reference"],
      retrieved_contexts=row["context_gen"]
  )
  context_recall_scores.append(context_recall_score.value)

In [ ]:
# Print the first 5 resutls and mean
print(context_recall_scores[0:5])
print(f"The mean context recall score is: {np.mean(context_recall_scores)}")

## Response Relevancy

In [ ]:
from ragas.metrics.collections import AnswerRelevancy

In [ ]:
# Define the response relevancy scorer
client = AsyncOpenAI(api_key = openai_api)
#client = OpenAI(api_key = openai_api)

eval_llm = llm_factory(
    model = "gpt-4o-mini",
    client = client)

eval_embeddings = embedding_factory(
    "openai",
    model = EMBEDDING_MODEL,
    client = client
)

response_relevancy_scorer = AnswerRelevancy(
    llm = eval_llm,
    embeddings = eval_embeddings)

In [ ]:
# Compute the response relevancies scores
response_relevancy_scores = []
for row in tqdm(test_df.to_dict(orient = "records"), desc = "Computing Response Relevancy Score...", total = len(test_df.to_dict(orient = "records"))):
  response_relevancy_score = await response_relevancy_scorer.ascore(
      user_input= row["user_input"],
      response = row["answer_gen"]
  )
  response_relevancy_scores.append(response_relevancy_score.value)